# Memory Spectroscopy

### Prerequisites

This guide assumes you have configured a `DeviceSetup` and created `BosonicQubit` objects with assigned parameters. Please see [our tutorials](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/index.html) if you need to create your setup and qubits for the first time.

You can run this notebook on real hardware in the lab. If you don't have the hardware at your disposal, you can also run the notebook "as is" using an emulated session (see below).

Before running the memory spectroscopy experiment, the following calibrations must have been performed:

- **Transmon spectroscopy**: the steady-state ancilla transmon resonance frequency `transmon_resonance_frequency_at_n0` must be known.
- **Transmon Rabi calibration**: the steady-state transmon $\pi$-pulse amplitude `selective_transmon_X180_amplitudes[0]` must be calibrated.

If you are just getting started with the LabOne Q Applications Library, please don't hesitate to reach out to us at info@zhinst.com.

### Background

In this how-to guide, you will perform a measurement to determine the resonance frequency of the high-Q **memory cavity** of a bosonic qubit using the `memory_spectroscopy` experiment workflow in the LabOne Q Applications Library.

#### The bosonic qubit system

A bosonic qubit encodes quantum information in the long-lived Fock states of a cavity (the *memory*). The control and readout of the memory is achieved through a dispersively coupled ancilla transmon. The dispersive interaction shifts the transmon resonance frequency by the dispersive shift $\chi$ for each photon present in the memory:

$$f_\mathrm{transmon}(n) = f_\mathrm{transmon}(0) + n \cdot \chi$$

where $n$ is the photon number in the memory cavity. The ancilla transmon is itself read out via a dedicated readout resonator using standard dispersive readout.

#### Memory spectroscopy via the ancilla transmon

Unlike a direct transmission measurement of the cavity, this experiment reads out the memory cavity state *indirectly* through the ancilla transmon. The pulse sequence is:

1. **Memory spectroscopy drive** — A long constant-amplitude tone is applied to the memory cavity drive line at a swept RF frequency $f_\mathrm{drive}$.
2. **Transmon $\pi$ pulse** — A $\pi$ pulse calibrated for the transmon when the memory is in the steady state, which is assumed to be the vacuum state $|0\rangle$ here, (i.e., at $f_\mathrm{transmon}(0)$) is applied.
3. **Readout** — The ancilla transmon is read out dispersively.

When $f_\mathrm{drive}$ is far from the memory resonance, the spectroscopy tone does not excite the cavity. The transmon $\pi$ pulse is then resonant and fully inverts the transmon: the readout signal is at its maximum. When $f_\mathrm{drive} \approx f_\mathrm{memory}$, the cavity is resonantly driven and photons build up. The resulting photon population shifts the transmon frequency away from $f_\mathrm{transmon}(0)$, causing the $\pi$ pulse to become off-resonant. The transmon is no longer fully inverted, producing a measurable dip in the readout signal. The frequency at which this dip occurs is the memory cavity resonance frequency $f_\mathrm{memory}$.

This technique is the bosonic-qubit analogue of using a probe tone to find a qubit frequency via its dispersive shift on the readout resonator.

### Imports

You'll start by importing `laboneq.simple`, as well as the `BosonicQubit` and `BosonicQubitOperations` classes from the bosonic qubits contribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from laboneq.simple import *

from laboneq_applications.contrib.qpu_types.bosonic_qubits import (
    BosonicQubit,
    BosonicQubitOperations,
)

### Define your experimental setup

Let's define our experimental setup. We will need:

- a [DeviceSetup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)
- one or more [BosonicQubits](https://docs.zhinst.com/labone_q_user_manual/applications_library/reference/contrib/index.html)
- a set of `BosonicQubitOperations`
- a [QPU](https://docs.zhinst.com/labone_q_user_manual/core/reference/dsl/quantum.html#laboneq.dsl.quantum.qpu.QPU)

Here we will be brief and mainly provide the code to obtain these objects. For full details, see the tutorials on [defining your experimental setup](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/getting_started.html) and [logbooks and data saving](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/logbooks.html).

We will use **2 bosonic qubits** in this guide. Change this number to match your setup.

#### Instrument serial numbers

In [ ]:
# Replace with the serial number of your SHFQC instrument
SHFQC_SERIAL = "DEV12073"

DATASERVER_HOST = "localhost"
DATASERVER_PORT = "8004"

number_of_qubits = 2

#### DeviceSetup

The memory spectroscopy experiment requires signal lines for the memory cavity drive, the ancilla transmon drive, and qubit readout. Here we use a single **SHFQC** instrument.

Each bosonic qubit uses:
- One **QA channel** pair (OUTPUT + INPUT) for readout
- One **SG channel** for the memory cavity drive (`drive_memory`)
- One **SG channel** for all number-selective transmon drives (`drive_transmon_at_n0` … `drive_transmon_at_n{max_photon_number}`)
- Optionally, one **SG channel** in LF mode for the SWAP drive (`swap_drive`)

If you already have a `DeviceSetup`, you can reuse it and skip this cell. Just make sure the signal line names match those expected by `BosonicQubit`.

In [ ]:
# Maximum photon number — determines how many selective transmon signal lines are created.
# Must match the value set in BosonicQubitParameters.max_photon_number below.
MAX_PHOTON_NUMBER = 5

device_setup = DeviceSetup(uid="my_bosonic_setup")
device_setup.add_dataserver(
    uid="my_bosonic_setup",
    host=DATASERVER_HOST,
    port=DATASERVER_PORT,
)

device_setup.add_instruments(
    SHFQC(
        uid="device_shfqc",
        address=SHFQC_SERIAL,
        device_options="SHFQC/LRT/PLUS/QC6CH/RTR",
    ),
)

# ── Signal lines for qubit q0 ─────────────────────────────────────────
device_setup.add_connections(
    "device_shfqc",
    # Readout (QA channel 0)
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
    # Memory cavity drive (SG channel 0)
    create_connection(to_signal="q0/drive_memory", ports="SGCHANNELS/0/OUTPUT"),
    # SWAP drive — optional, used for Fock-state preparation (SG channel 1)
    create_connection(to_signal="q0/swap_drive", ports="SGCHANNELS/4/OUTPUT"),
)
# Photon-number-selective transmon drive lines for q0 (all share SG channel 2)
for n in range(MAX_PHOTON_NUMBER + 1):
    device_setup.add_connections(
        "device_shfqc",
        create_connection(
            to_signal=f"q0/drive_transmon_at_n{n}",
            ports="SGCHANNELS/2/OUTPUT",
        ),
    )

# ── Signal lines for qubit q1 (optional) ─────────────────────────────
if number_of_qubits == 2:
    device_setup.add_connections(
        "device_shfqc",
        create_connection(to_signal="q1/measure", ports="QACHANNELS/0/OUTPUT"),
        create_connection(to_signal="q1/acquire", ports="QACHANNELS/0/INPUT"),
        create_connection(to_signal="q1/drive_memory", ports="SGCHANNELS/1/OUTPUT"),
        create_connection(to_signal="q1/swap_drive", ports="SGCHANNELS/5/OUTPUT"),
    )
    for n in range(MAX_PHOTON_NUMBER + 1):
        device_setup.add_connections(
            "device_shfqc",
            create_connection(
                to_signal=f"q1/drive_transmon_at_n{n}",
                ports="SGCHANNELS/3/OUTPUT",
            ),
        )

#### BosonicQubits

We will create `BosonicQubit` objects from the logical signal groups in our `DeviceSetup` and assign qubit parameters. The qubit UID will match the name of the logical signal group (`q0`, `q1`, …).

Adjust the parameter values to reflect the properties of your device. The parameters relevant to memory spectroscopy are highlighted in the comments.

In [ ]:
# Qubit parameters — adjust these to match your device.
# Parameters marked (*) are directly relevant to the memory spectroscopy experiment.
QUBIT_PARAMS = {
    # ── Memory cavity (*) ────────────────────────────────────────────
    "memory_lo_frequency": 5.8e9,  # LO for memory cavity drive
    "memory_resonance_frequency": 6.0e9,  # (*) Expected memory resonance (RF)
    "memory_drive_amplitude": 0.5,
    "memory_drive_length": 0.8e-6,
    "memory_drive_pulse": {"function": "gaussian", "sigma": 0.25},
    "memory_spectroscopy_amplitude": 0.5,  # (*) Amplitude of the spectroscopy tone
    "memory_spectroscopy_length": 4e-6,  # (*) Duration of the spectroscopy tone
    "memory_drive_range": 10,
    # ── Ancilla transmon ──────────────────────────────────────────────
    "transmon_lo_frequency": 4.8e9,
    "transmon_resonance_frequency_at_n0": 5.0e9,  # (*) Transmon freq when memory in |0>
    "transmon_drive_range": 10,
    # ── Selective transmon Rx drives (*) ─────────────────────────────
    # These are used to apply the pi pulse after the spectroscopy tone.
    "max_photon_number": MAX_PHOTON_NUMBER,
    "selective_transmon_Rx_length": 0.08e-6,
    "selective_transmon_Rx_pulse": {"function": "gaussian", "sigma": 0.25},
    "selective_transmon_X180_amplitudes": [0.8]
    * (MAX_PHOTON_NUMBER + 1),  # (*) pi amplitudes
    "selective_transmon_X90_amplitudes": [0.4] * (MAX_PHOTON_NUMBER + 1),
    # ── Readout ───────────────────────────────────────────────────────
    "readout_lo_frequency": 6.8e9,
    "readout_resonator_frequency": 7.0e9,
    "readout_amplitude": 0.5,
    "readout_length": 1e-6,
    "readout_pulse": {"function": "const"},
    "readout_integration_length": 1e-6,
    "readout_integration_delay": 20e-9,
    # ── Reset ─────────────────────────────────────────────────────────
    "reset_delay_length": 1e-6,
    # ── Memory-transmon coupling (*) ──────────────────────────────────
    "chi": -1e6,  # (*) Dispersive shift in Hz
    # ── Displacement calibration result (not needed for this experiment) ──
    "displacement_amp_per_unit_beta": None,
}

In [ ]:
q0 = BosonicQubit.from_logical_signal_group(
    uid="q0",
    lsg=device_setup.logical_signal_groups["q0"],
    parameters=QUBIT_PARAMS,
)
qubits = [q0]

if number_of_qubits == 2:
    q1 = BosonicQubit.from_logical_signal_group(
        uid="q1",
        lsg=device_setup.logical_signal_groups["q1"],
        parameters=QUBIT_PARAMS,
    )
    qubits.append(q1)

for q in qubits:
    print("-------------")
    print("Qubit UID:", q.uid)
    print("Qubit logical signals:")
    for sig, lsg in q.signals.items():
        print(f"  {sig:<30} ('{lsg}')")

#### Quantum Operations

Create the set of `BosonicQubitOperations`:

In [ ]:
qops = BosonicQubitOperations()

#### QPU

Create the `QPU` object from the qubits and the quantum operations:

In [ ]:
from laboneq.dsl.quantum import QPU

qpu = QPU(qubits, quantum_operations=qops)

### Connect to Session

In [ ]:
session = Session(device_setup)
session.connect(do_emulation=True)  # set do_emulation=False when at a real setup

### Create a `FolderStore` for Saving Data

The experiment `Workflows` can automatically save the inputs and outputs of all their tasks to the folder path we specify when instantiating the `FolderStore`. Here, we choose the current working directory.

In [ ]:
from pathlib import Path

folder_store = workflow.logbook.FolderStore(Path.cwd() / "data")

In [ ]:
# We disable saving in this guide. To enable it, simply run folder_store.activate().
folder_store.deactivate()

### Optional: Configure the LoggingStore

You can also activate/deactivate the `LoggingStore`, which controls the display of `Workflow` logging information in the notebook. See the [tutorial on Recording Experiment Workflow Results](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/logbooks.html) for details.

Displaying the `Workflow` logging information is activated by default, but here we deactivate it to shorten the outputs, which are not very meaningful in emulation mode.

**We recommend that you do not deactivate the Workflow logging in practice.**

In [ ]:
from laboneq.workflow.logbook import LoggingStore

logging_store = LoggingStore()
logging_store.deactivate()

### Running the Experiment Workflow

You'll now instantiate the experiment workflow and run it. For more details on what experiment workflows are and what tasks they execute, see the [Experiment Workflows tutorial](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/experiment_workflows.html).

Start by importing the `memory_spectroscopy` experiment workflow from the bosonic qubits contribution, as well as `plot_simulation` for inspecting the pulse sequence.

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

from laboneq_applications.contrib.experiments.bosonic_qubits import memory_spectroscopy

#### Configure the experiment options

Let's create the options object for the memory spectroscopy experiment and inspect it using `workflow.show_fields`:

In [ ]:
options = memory_spectroscopy.experiment_workflow.options()
workflow.show_fields(options)

Notice that, unless we change it:

- The experiment is run in `AcquisitionType.INTEGRATION` and `AveragingMode.CYCLIC`, using 1024 averages (`count`).
- There is no analysis workflow for this experiment: plotting and fitting of the memory cavity spectrum is performed by the user on the raw result data (see below).

The most relevant option to adjust for memory spectroscopy is typically the number of averages (`count`. Here, let's reduce the number of averages for a quicker run in emulation mode:

In [ ]:
options.count(16)

#### Define the frequency sweep

We sweep the RF frequency of the memory cavity drive around the expected memory cavity resonance. The sweep range should be wide enough to capture the resonance dip. A good starting point is $\pm 100$ MHz around the expected resonance frequency.

In [ ]:
# Frequency sweep for each qubit (RF frequencies in Hz)
qubit = qpu.quantum_elements[0]

frequencies_q0 = np.linspace(
    qubit.parameters.memory_resonance_frequency - 100e6,
    qubit.parameters.memory_resonance_frequency + 100e6,
    1001,
)

# Run on a single qubit
exp_workflow = memory_spectroscopy.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit.uid],
    frequencies=[frequencies_q0],
    options=options,
)

workflow_results = exp_workflow.run()

#### Inspect the Tasks That Were Run

The memory spectroscopy workflow executes three tasks in sequence: `create_experiment`, `compile_experiment`, and `run_experiment`. There is no built-in analysis task — the raw data is analysed by the user.

In [ ]:
for t in workflow_results.tasks:
    print(t)

#### Inspect the Output Simulation

You can inspect the compiled experiment and plot the simulated output to verify that the pulse sequence looks as expected. The memory spectroscopy sequence consists of a long spectroscopy tone on the memory cavity drive line, followed by a short transmon $\pi$ pulse and a readout pulse.

The total duration of one repetition is approximately:

$$T \approx t_\mathrm{spec} + t_{\pi} + t_\mathrm{readout} + t_\mathrm{reset}$$

In [ ]:
q = qpu.quantum_elements[0]
total_time = (
    q.parameters.memory_spectroscopy_length
    + q.parameters.selective_transmon_Rx_length
    + q.parameters.readout_length
    + q.parameters.reset_delay_length
)

compiled_experiment = workflow_results.tasks["compile_experiment"].output
plot_simulation(
    compiled_experiment,
    length=total_time * 1.05,
)

#### Inspect the Source Code of the Pulse-Sequence Creation Task

You can inspect the source code of the `create_experiment` task to see exactly how the pulse sequence is constructed using quantum operations. To learn more about quantum operations, see the [Quantum Operations tutorial](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/quantum_operations.html).

In [ ]:
memory_spectroscopy.create_experiment.src

#### Access the Raw Results

The experiment result is a complex-valued array (I/Q data) with one point per frequency step. You can access it via the `run_experiment` task output, using the result handle for each qubit.

The memory cavity resonance appears as a dip in the magnitude (or a change in phase) at the frequency $f_\mathrm{drive} \approx f_\mathrm{memory}$. Near resonance, the cavity is excited, the ancilla transmon frequency is shifted by $\chi$, and the $\pi$ pulse becomes off-resonant — reducing the readout signal.

In [ ]:
run_result = workflow_results.tasks["run_experiment"].output
qubit_uid = qpu.quantum_elements[0].uid

# Access the complex I/Q data for q0
iq_data = run_result.data.q0.result.data
magnitude = np.abs(iq_data)

plt.figure(figsize=(8, 4))
plt.plot(frequencies_q0 / 1e9, magnitude, color="steelblue")
plt.xlabel("Memory drive frequency (GHz)")
plt.ylabel("|S| (a.u.)")
plt.title(f"Memory Spectroscopy — {qubit_uid}")
plt.tight_layout()
plt.show()

print("Number of frequency points:", len(magnitude))
print(
    "Expected resonance at:", qubit.parameters.memory_resonance_frequency / 1e9, "GHz"
)

## Parallel Memory Spectroscopy

The `memory_spectroscopy` workflow natively supports running multiple qubits in parallel. When you pass a list of qubit UIDs together with a corresponding list of frequency arrays, the pulse sequences for all qubits are compiled into a single experiment and executed simultaneously on the hardware — halving experiment time compared to sequential single-qubit runs.

Each qubit gets its own independently chosen frequency sweep, so qubits with different memory resonance frequencies can still be characterised in one shot. The only constraint is that all frequency arrays must have the **same number of points**, because they are swept in lock-step by the hardware.

#### Run the two-qubit experiment

Pass both qubit UIDs and their frequency arrays to the workflow:

In [ ]:
q0 = qpu.quantum_elements[0]
q1 = qpu.quantum_elements[1]

# Each qubit gets its own frequency sweep, independently centred on its expected resonance.
frequencies_q0 = np.linspace(
    q0.parameters.memory_resonance_frequency - 100e6,
    q0.parameters.memory_resonance_frequency + 100e6,
    1001,
)
frequencies_q1 = np.linspace(
    q1.parameters.memory_resonance_frequency - 10e6,
    q1.parameters.memory_resonance_frequency + 10e6,
    1001,
)

exp_workflow_2q = memory_spectroscopy.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[q0.uid, q1.uid],
    frequencies=[frequencies_q0, frequencies_q1],
    options=options,
)

workflow_results_2q = exp_workflow_2q.run()

#### Plot the Results

Access the complex I/Q data for each qubit and plot the memory spectra side by side. As in the single-qubit case, the memory cavity resonance appears as a dip in the magnitude at $f_\mathrm{drive} \approx f_\mathrm{memory}$.

In [ ]:
run_result_2q = workflow_results_2q.tasks["run_experiment"].output

iq_q0 = run_result_2q.data.q0.result.data
iq_q1 = run_result_2q.data.q1.result.data

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, iq_data, freqs, q in zip(
    axes,
    [iq_q0, iq_q1],
    [frequencies_q0, frequencies_q1],
    [q0, q1],
    strict=False,
):
    ax.plot(freqs / 1e9, np.abs(iq_data), color="steelblue")
    ax.set_xlabel("Memory drive frequency (GHz)")
    ax.set_ylabel("|S| (a.u.)")
    ax.set_title(f"Memory Spectroscopy — {q.uid}")

plt.tight_layout()
plt.show()

Great! You've now run your memory spectroscopy experiment. The next step in the bosonic qubit tune-up sequence is the **displacement calibration**, which maps the memory drive amplitude to the phase-space displacement amplitude $|\beta|$.